<a href="https://colab.research.google.com/github/RushabhMowade/Currency_Conversion_Tool/blob/main/Currency_conversion_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
pip install langchain-openai

In [43]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [44]:
!pip install -U langchain-google-genai

In [45]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Gemini key: ")

Gemini key: ··········


In [46]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f"https://v6.exchangerate-api.com/v6/fca7817d4994179d4c09e199/pair/{base_currency}/{target_currency}"
  response = requests.get(url)
  return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [47]:
get_conversion_factor.args


{'base_currency': {'title': 'Base Currency', 'type': 'string'},
 'target_currency': {'title': 'Target Currency', 'type': 'string'}}

In [48]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1786838401,
 'time_last_update_utc': 'Sun, 16 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1786924801,
 'time_next_update_utc': 'Mon, 17 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.5035}

In [60]:
convert.invoke({'base_currency_value':10, 'conversion_rate':95.50})

955.0

# tool binding

In [50]:

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [51]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [52]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [53]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [54]:
ai_message = llm_with_tools.invoke(messages)


In [55]:
messages.append(ai_message)

In [56]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'ff6e3fea-4d77-46d9-9d80-dd8d4c896211',
  'type': 'tool_call'}]

In [62]:
import json
from langchain_core.messages import ToolMessage

# Initialize a variable to hold the current LLM response, starting with the one from the first invoke.
# This 'ai_message' is from cell M5o203uJKUZO.
current_llm_response = ai_message

# Keep track of conversion_rate if needed across tool calls.
conversion_rate = None

# Loop as long as the current LLM response suggests tool calls
while current_llm_response.tool_calls:
    for tool_call in current_llm_response.tool_calls:
        if tool_call['name'] == 'get_conversion_factor':
            tool_output_invoke = get_conversion_factor.invoke(tool_call['args'])

            # tool_output_invoke is already a dictionary, converting it to a string for ToolMessage content
            tool_output_content = json.dumps(tool_output_invoke)

            # Extracting the conversion rate directly from the dictionary
            conversion_rate = tool_output_invoke['conversion_rate']

            messages.append(ToolMessage(tool_output_content, name=tool_call['name'], tool_call_id=tool_call['id']))

        elif tool_call['name'] == 'convert':
            # arguments
            args_for_convert = tool_call['args'].copy()
            if conversion_rate is not None:
                args_for_convert['conversion_rate'] = conversion_rate
            else:
                raise ValueError("Conversion rate not available for 'convert' tool.")

            tool_output_invoke = convert.invoke(args_for_convert)

            # The content from convert.invoke is a float
            tool_output_content = str(tool_output_invoke)
            messages.append(ToolMessage(tool_output_content, name=tool_call['name'], tool_call_id=tool_call['id']))

        else:
            print(f"Warning: Encountered unknown tool: {tool_call['name']}")
            break # Exit the loop of tool_calls for the current response

    current_llm_response = llm_with_tools.invoke(messages)
    messages.append(current_llm_response)
final_answer_content = current_llm_response.content

In [63]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "INR", "target_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'ff6e3fea-4d77-46d9-9d80-dd8d4c896211': 'CuYGARFNMg9YwpunjleE3ZseLsMPqKSwDplrzE9N7zpP+H7G5QBwJ0vXxeNeCYHJzeMQ+CYaMuX9TodC6f/xEIs5PF51u2NlzGA+AwlLXiNDwk4BTkdMl1TLFvvrEX5KlyhWNyJAca2UKQZESE+9LxWyUWnyuAinIsifQPwA9B/4TWHu8/nzZQxpE6vp4RuGwtExnsHuswMiDTQIwCOvntIdh3PjxabL0P9tfUb30QO1QAwJCWCZC/VJjE6LOOHY+OLTgjX2sR2lHGRuLdM+MYHO4DEjAc1tOhqv35teATzrcHRQPs3ArQ9hIneNLEyugPC2r0FkEKYUq7QOUkD8BEzUcUG7yYp+y/Xsjr9MEE/9ltmW9NCtXdDlE9j6vOAIzQWIfOKwDfs7d54z5iIX7ORc4+aIfXr9aWltoSzCj7+0Aq7u8eT8kpOmsW0VVYNXnsxq1VDUX8btX357a+Iz5KyKITcxDcXA+m6+pT8uv43lynh7wxHPOLotp/CzrHYj4CZgxxsqEzPelT7uLQpzSuGnNSagxfSV84BiaRaxlCgtQnHa9vHNjaaWfrKj4hmHFqi+L

In [67]:
print(final_answer_content)

The conversion factor between INR and USD is 0.01047.
10 INR is equal to 0.1047 USD.
